In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mutual_info_score
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors

DATA_PATH = Path("Data/dataset.csv")
OUTPUT_DIR = Path("Data/processed")
TARGET = "risk_level"
TEST_SIZE = 0.30
RANDOM_STATE = 42
N_SELECTED = 13
RELIEFF_K = 10

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
df = df.drop_duplicates().reset_index(drop=True)

if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' not found.")

df = df.dropna(subset=[TARGET]).reset_index(drop=True)
print("Dataset shape:", df.shape)
print("Target distribution:\n", df[TARGET].value_counts())
display(df.head())

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(str).str.strip()

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y_encoded, test_size=TEST_SIZE, stratify=y_encoded,
    random_state=RANDOM_STATE
)

X_train_raw = X_train_raw.copy().ffill()
X_test_raw = X_test_raw.copy().ffill()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

for c in num_cols:
    mean_value = X_train_raw[c].mean()
    X_train_raw[c] = X_train_raw[c].fillna(mean_value)
    X_test_raw[c] = X_test_raw[c].fillna(mean_value)

for c in cat_cols:
    mode_value = X_train_raw[c].mode()
    fill_value = mode_value.iloc[0] if not mode_value.empty else "Unknown"
    X_train_raw[c] = X_train_raw[c].fillna(fill_value)
    X_test_raw[c] = X_test_raw[c].fillna(fill_value)

X_train = pd.get_dummies(X_train_raw, columns=cat_cols, dtype=float)
X_test = pd.get_dummies(X_test_raw, columns=cat_cols, dtype=float)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

scaler = MinMaxScaler(feature_range=(0, 1))
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_train.columns, index=X_test.index)

y_train = np.asarray(y_train)
y_test = np.asarray(y_test)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Encoded features:", X_train.shape[1])
print("Target classes:", list(label_encoder.classes_))

In [ ]:
def reliefF_score(X, y, k=10):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    n, p = X.shape
    k = min(k, n - 1)
    D = pairwise_distances(X)
    classes, counts = np.unique(y, return_counts=True)
    priors = dict(zip(classes, counts / n))
    scores = np.zeros(p)

    for i in range(n):
        same = np.where(y == y[i])[0]
        same = same[same != i]
        hits = same[np.argsort(D[i, same])[:k]]

        if len(hits):
            scores -= np.mean(np.abs(X[i] - X[hits]), axis=0) / n

        for cls in classes:
            if cls == y[i]:
                continue
            miss_idx = np.where(y == cls)[0]
            misses = miss_idx[np.argsort(D[i, miss_idx])[:k]]
            if len(misses):
                weight = priors[cls] / (1 - priors[y[i]])
                scores += weight * np.mean(np.abs(X[i] - X[misses]), axis=0) / n

    return scores


def mrmr_score(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    relevance = np.array([mutual_info_score(y, pd.cut(X[:, j], bins=10, duplicates="drop").codes)
                         for j in range(X.shape[1])])
    redundancy = np.abs(np.corrcoef(X, rowvar=False))
    redundancy = np.nan_to_num(redundancy, nan=0.0)
    np.fill_diagonal(redundancy, 0)
    return relevance - redundancy.mean(axis=1)


def fisher_score(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    overall_mean = X.mean(axis=0)
    numerator = np.zeros(X.shape[1])
    denominator = np.zeros(X.shape[1])

    for cls in np.unique(y):
        X_cls = X[y == cls]
        cls_mean = X_cls.mean(axis=0)
        numerator += len(X_cls) * (cls_mean - overall_mean) ** 2
        denominator += np.sum((X_cls - cls_mean) ** 2, axis=0)

    return numerator / (denominator + 1e-10)


def laplacian_score(X, k=5):
    X = np.asarray(X, dtype=float)
    n, p = X.shape
    k = min(k, n - 1)
    distances = pairwise_distances(X)
    np.fill_diagonal(distances, np.inf)
    neighbors = np.argsort(distances, axis=1)[:, :k]
    W = np.zeros((n, n))

    for i in range(n):
        for j in neighbors[i]:
            W[i, j] = np.exp(-distances[i, j] ** 2)
            W[j, i] = W[i, j]

    D = W.sum(axis=1)
    L = np.diag(D) - W
    scores = np.zeros(p)

    for j in range(p):
        f = X[:, j]
        weighted_mean = np.sum(D * f) / (D.sum() + 1e-10)
        f = f - weighted_mean
        scores[j] = (f @ L @ f) / (f @ np.diag(D) @ f + 1e-10)

    return scores

In [ ]:
feature_names = X_train.columns.tolist()
X_train_np = X_train.to_numpy()
y_train_np = y_train

relief_scores = reliefF_score(X_train_np, y_train_np, k=RELIEFF_K)
mrmr_scores = mrmr_score(X_train_np, y_train_np)
fisher_scores = fisher_score(X_train_np, y_train_np)
laplacian_scores = laplacian_score(X_train_np)

scores_df = pd.DataFrame({
    "Feature": feature_names,
    "ReliefF": relief_scores,
    "mRMR": mrmr_scores,
    "Fisher": fisher_scores,
    "Laplacian": laplacian_scores
})

scores_df["ReliefF_Rank"] = scores_df["ReliefF"].rank(ascending=False, method="min")
scores_df["mRMR_Rank"] = scores_df["mRMR"].rank(ascending=False, method="min")
scores_df["Fisher_Rank"] = scores_df["Fisher"].rank(ascending=False, method="min")
scores_df["Laplacian_Rank"] = scores_df["Laplacian"].rank(ascending=True, method="min")

rank_cols = ["ReliefF_Rank", "mRMR_Rank", "Fisher_Rank", "Laplacian_Rank"]
scores_df["Hybrid_Rank"] = scores_df[rank_cols].mean(axis=1)
scores_df = scores_df.sort_values("Hybrid_Rank").reset_index(drop=True)

selected_features = scores_df["Feature"].head(min(N_SELECTED, len(feature_names))).tolist()

X_train_selected = X_train[selected_features].copy()
X_test_selected = X_test[selected_features].copy()

print("Selected features:", len(selected_features))
print(selected_features)
display(scores_df.head(N_SELECTED))

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_processed = X_train_selected.copy()
train_processed[TARGET] = y_train

test_processed = X_test_selected.copy()
test_processed[TARGET] = y_test

train_processed.to_csv(OUTPUT_DIR / "train_processed.csv", index=False)
test_processed.to_csv(OUTPUT_DIR / "test_processed.csv", index=False)
scores_df.to_csv(OUTPUT_DIR / "feature_selection_scores.csv", index=False)

with open(OUTPUT_DIR / "selected_features.json", "w") as f:
    json.dump(selected_features, f, indent=4)

with open(OUTPUT_DIR / "target_classes.json", "w") as f:
    json.dump({str(i): cls for i, cls in enumerate(label_encoder.classes_)}, f, indent=4)

print("Preprocessing completed.")
print("Train shape:", train_processed.shape)
print("Test shape:", test_processed.shape)
print("Saved to:", OUTPUT_DIR.resolve())
print("Selected features:", selected_features)